
  <img src="https://github.com/gefero/factor_data_tuto_NLP_SICSS/blob/main/imgs/logo_final_conjunto.png?raw=true" width="80%">


# Summer Institute in Computational Social Sciences - Buenos Aires 2026
# Taller: Procesamiento de Lenguaje Natural y polarización
# Clasificación de tweets con distintas representaciones: TF, TF-IDF y Word Embeddings

# Introducción

El objetivo de este notebook es comparar distintas formas de representar texto como *features* para un modelo de clasificación, usando como caso el dataset **HatEval** (SemEval-2019 Task 5), que ya vienen utilizando en las prácticas de este taller. El dataset viene dividido en tres splits, cada uno en su propio archivo dentro de `data/`:

- `hateval_train_df.csv`: split de **entrenamiento**. Lo usamos para ajustar (`fit`) cada modelo.
- `hateval_dev_df.csv`: split de **validación**. Lo usamos para elegir el mejor valor del hiperparámetro de regularización de cada modelo.
- `hateval_test_df.csv`: split de **test**. Lo usamos únicamente al final, para evaluar el desempeño de cada modelo sobre datos que no participaron ni del entrenamiento ni de la elección de hiperparámetros.

Cada fila de estos archivos es un tweet con las siguientes columnas relevantes:

- `id`: identificador del tweet
- `text`: el texto del tweet
- `language`: idioma del tweet (`en` o `es`)
- `HS`: 1 si el tweet contiene discurso de odio (*hate speech*), 0 en caso contrario
- `TR`, `AG`: otras anotaciones (agresividad, si el odio está dirigido a un individuo o a un grupo) que no vamos a usar acá

La tarea de clasificación va a ser predecir `HS` (discurso de odio sí/no) a partir del texto del tweet.

Como en la última sección vamos a usar **embeddings preentrenados en español** (SBWCE), y los tres splits tienen tweets tanto en inglés como en español, nos vamos a quedar solamente con los tweets en español (`language == 'es'`) de cada split para poder comparar las tres representaciones sobre los mismos datos.

Al igual que en el ejemplo con reseñas de Amazon (`cap0/ejemplo_clasificacion.ipynb`), vamos a entrenar en cada caso una regresión logística regularizada por LASSO (penalización L1), variando el hiperparámetro de regularización $C$ (a menor $C$, mayor regularización). A diferencia de ese ejemplo, acá **no** usamos validación cruzada (K-Fold) para elegir $C$: como ya contamos con un split de validación (`dev`) independiente, entrenamos cada candidato sobre `train` y elegimos el que mejor performa sobre `dev`. `test` queda completamente afuera de ese proceso, y solo se usa al final para reportar el desempeño de cada modelo ya elegido.

Vamos a comparar tres formas de vectorizar los tweets:

1. **TF** (*Term Frequency*, bolsa de palabras con conteos)
2. **TF-IDF** (*Term Frequency - Inverse Document Frequency*)
3. **Word embeddings preentrenados** (promedio de vectores de palabras)

Además de las métricas de desempeño (ROC AUC, accuracy, precision, recall, F1), vamos a comparar el **costo computacional** de cada representación: cuánto tarda en entrenarse (vectorizar + ajustar el clasificador) y cuánto tarda en predecir sobre un tweet nuevo. No es un detalle menor: como vamos a ver, la representación que mejor predice no es necesariamente la más barata de entrenar o servir.

In [ ]:
## Ejecutar para descargar los embeddings preentrenados en español (SBWCE)
!wget -P ./models https://cs.famaf.unc.edu.ar/~ccardellino/SBWCE/SBW-vectors-300-min5.bin.gz && gunzip ./models/SBW-vectors-300-min5.bin.gz
!pip install gensim
!git clone https://github.com/gefero/factor_data_tuto_NLP_SICSS.git

# TF con LASSO

## Preparación de los datos

Cargamos los tres splits de HatEval (`hateval_train_df.csv`, `hateval_dev_df.csv` y `hateval_test_df.csv`), nos quedamos en cada uno con los tweets en español y preprocesamos el texto:

1. Convertimos a minúsculas
2. Eliminamos URLs y menciones (`@usuario`), que son ruido propio de los tweets
3. Eliminamos signos de puntuación
4. Reemplazamos números por la palabra `DIGITO`
5. Eliminamos acentos y caracteres no ASCII

Para cada split, `X` va a ser el texto preprocesado y `y` la variable binaria `HS` (discurso de odio).

In [ ]:
# Importamos las librerías necesarias
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import re
import unicodedata
import time
import warnings
warnings.filterwarnings('ignore')

In [ ]:
train_path = './factor_data_tuto_NLP_SICSS/data/hateval_train_df.csv'
dev_path = './factor_data_tuto_NLP_SICSS/data/hateval_dev_df.csv'
test_path = './factor_data_tuto_NLP_SICSS/data/hateval_test_df.csv'

In [ ]:
# Función para preprocesar el texto
def preprocess_text(text):
    # Convertir a minúsculas
    text = text.lower()

    # Eliminar URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Eliminar menciones (@usuario)
    text = re.sub(r'@\w+', ' ', text)

    # Reemplazar puntuación
    text = re.sub(r'[^\w\s]', ' ', text)

    # Reemplazar números por 'DIGITO'
    text = re.sub(r'\d+', 'DIGITO', text)

    # Reemplazar caracteres no ASCII (acentos, etc.)
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8')

    # Colapsar espacios múltiples
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Cargamos un split y nos quedamos con los tweets en español
def load_hateval_split(path):
    df = pd.read_csv(path)
    df = df[df['language'] == 'es'].reset_index(drop=True)
    df['text_clean'] = df['text'].apply(preprocess_text)
    return df

train_df = load_hateval_split(train_path)
dev_df = load_hateval_split(dev_path)
test_df = load_hateval_split(test_path)

# Preparamos las variables para el modelo
X_train, y_train = train_df['text_clean'], train_df['HS']
X_dev, y_dev = dev_df['text_clean'], dev_df['HS']
X_test, y_test = test_df['text_clean'], test_df['HS']

print(f"Train: {len(X_train)} tweets | Dev: {len(X_dev)} tweets | Test: {len(X_test)} tweets")

## Búsqueda de hiperparámetros usando el split de validación

Armamos un pipeline con:

1. **Vectorización TF**: `CountVectorizer` con ngramas de 1 y 2 palabras
2. **Regresión logística LASSO** (penalización L1, solver `liblinear`)

Para elegir el mejor valor de `C` (inverso de la fuerza de regularización) probamos 30 valores en escala logarítmica entre $10^{-10}$ y $10^{1}$: para cada uno, entrenamos el pipeline sobre `train` y calculamos el ROC AUC sobre `dev`. Nos quedamos con el `C` que da mejor ROC AUC en `dev`.

También cronometramos esta búsqueda completa (`tuning_time_s`): como el vectorizador vive *adentro* del `Pipeline`, cada uno de los 30 candidatos re-vectoriza `train` desde cero, así que este tiempo pesa bastante más que el de un solo `fit` del modelo final.

In [ ]:
# Creamos una grilla de valores para el parámetro C (inverso de la penalización)
C_values = np.logspace(-10, 1, 30)

# Pipeline con TF y LASSO
pipeline_tf = Pipeline([
    ('vectorizer', CountVectorizer(
        ngram_range=(1, 2),
        token_pattern=r'\b\w+\b'
    )),
    ('classifier', LogisticRegression(
        penalty='l1',
        solver='liblinear',
        random_state=234
    ))
])

# Almacenaremos los resultados de la búsqueda aquí
val_results = []

In [ ]:
# Cronometramos la búsqueda completa: como el vectorizador vive adentro del
# Pipeline, cada candidato de C re-vectoriza train desde cero (30 veces en
# total), así que este tiempo es mucho mayor que el de un solo fit
t0 = time.perf_counter()
for C in C_values:
    pipeline_tf.set_params(classifier__C=C)

    # Entrenamos sobre train y evaluamos sobre dev
    pipeline_tf.fit(X_train, y_train)
    dev_proba = pipeline_tf.predict_proba(X_dev)[:, 1]

    val_results.append({
        'C': C,
        'roc_auc_dev': roc_auc_score(y_dev, dev_proba)
    })
tuning_time_tf = time.perf_counter() - t0

# Convertimos resultados a DataFrame
val_results_df = pd.DataFrame(val_results)

# Encontramos el mejor C
best_idx = val_results_df['roc_auc_dev'].idxmax()
best_C_tf = val_results_df.loc[best_idx, 'C']

print(f"Tiempo de búsqueda (30 valores de C): {tuning_time_tf:.1f}s")
print(f"Mejor valor de C: {best_C_tf:.6f}")
print(f"Mejor ROC AUC (dev): {val_results_df.loc[best_idx, 'roc_auc_dev']:.3f}")

## Modelo final

Entrenamos el pipeline final sobre `train` con el mejor valor de `C` encontrado y evaluamos sobre `test` (que no participó ni del entrenamiento ni de la elección de hiperparámetros) con ROC AUC, accuracy, precision, recall y F1.

Además cronometramos `fit` (tiempo de entrenamiento) y `predict` (tiempo de inferencia) del pipeline completo: como el vectorizador vive adentro del `Pipeline`, esos tiempos ya incluyen tanto vectorizar el texto como ajustar/predecir con la regresión logística.

In [ ]:
# Ajustamos el modelo final con el mejor C
final_pipeline_tf = Pipeline([
    ('vectorizer', CountVectorizer(
        ngram_range=(1, 2),
        token_pattern=r'\b\w+\b'
    )),
    ('classifier', LogisticRegression(
        C=best_C_tf,
        penalty='l1',
        solver='liblinear',
        random_state=234
    ))
])

# Entrenamiento: incluye vectorizar train y ajustar el clasificador
t0 = time.perf_counter()
final_pipeline_tf.fit(X_train, y_train)
train_time_tf = time.perf_counter() - t0

# Inferencia: incluye vectorizar test y predecir
t0 = time.perf_counter()
y_pred = final_pipeline_tf.predict(X_test)
inference_time_tf = time.perf_counter() - t0

# predict_proba queda fuera del bloque cronometrado: solo lo usamos para el
# ROC AUC, y volvería a pagar el costo de vectorizar test
y_pred_proba = final_pipeline_tf.predict_proba(X_test)[:, 1]

# Métricas finales
results_tf = {
    'roc_auc': roc_auc_score(y_test, y_pred_proba),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'tuning_time_s': tuning_time_tf,
    'train_time_s': train_time_tf,
    'total_time_s': tuning_time_tf + train_time_tf,
    'inference_time_s': inference_time_tf,
    'inference_ms_per_tweet': inference_time_tf / len(X_test) * 1000
}

for metric, value in results_tf.items():
    print(f"{metric}: {value:.3f}")

# TF-IDF con LASSO

A diferencia de TF (que solo cuenta ocurrencias), **TF-IDF** pondera cada término según qué tan frecuente es dentro de un tweet (TF) pero penalizando los términos que aparecen en muchos tweets distintos (IDF). Esto suele darle más peso a palabras discriminativas y menos a palabras muy comunes.

Reutilizamos los mismos `X_train`/`y_train`, `X_dev`/`y_dev` y `X_test`/`y_test` de la sección anterior, y repetimos el mismo procedimiento (elegir `C` con el split de validación y evaluar sobre test) pero con `TfidfVectorizer` en lugar de `CountVectorizer`.

In [ ]:
# Pipeline con TF-IDF y LASSO
pipeline_tfidf = Pipeline([
    ('vectorizer', TfidfVectorizer(
        ngram_range=(1, 2),
        token_pattern=r'\b\w+\b'
    )),
    ('classifier', LogisticRegression(
        penalty='l1',
        solver='liblinear',
        random_state=234
    ))
])

val_results = []

In [ ]:
# Igual que en TF: el vectorizador vive adentro del Pipeline, así que cada
# candidato de C re-vectoriza train desde cero
t0 = time.perf_counter()
for C in C_values:
    pipeline_tfidf.set_params(classifier__C=C)

    pipeline_tfidf.fit(X_train, y_train)
    dev_proba = pipeline_tfidf.predict_proba(X_dev)[:, 1]

    val_results.append({
        'C': C,
        'roc_auc_dev': roc_auc_score(y_dev, dev_proba)
    })
tuning_time_tfidf = time.perf_counter() - t0

val_results_df = pd.DataFrame(val_results)

best_idx = val_results_df['roc_auc_dev'].idxmax()
best_C_tfidf = val_results_df.loc[best_idx, 'C']

print(f"Tiempo de búsqueda (30 valores de C): {tuning_time_tfidf:.1f}s")
print(f"Mejor valor de C: {best_C_tfidf:.6f}")
print(f"Mejor ROC AUC (dev): {val_results_df.loc[best_idx, 'roc_auc_dev']:.3f}")

In [ ]:
# Ajustamos el modelo final con el mejor C
final_pipeline_tfidf = Pipeline([
    ('vectorizer', TfidfVectorizer(
        ngram_range=(1, 2),
        token_pattern=r'\b\w+\b'
    )),
    ('classifier', LogisticRegression(
        C=best_C_tfidf,
        penalty='l1',
        solver='liblinear',
        random_state=234
    ))
])

# Entrenamiento: incluye vectorizar train y ajustar el clasificador
t0 = time.perf_counter()
final_pipeline_tfidf.fit(X_train, y_train)
train_time_tfidf = time.perf_counter() - t0

# Inferencia: incluye vectorizar test y predecir
t0 = time.perf_counter()
y_pred = final_pipeline_tfidf.predict(X_test)
inference_time_tfidf = time.perf_counter() - t0

y_pred_proba = final_pipeline_tfidf.predict_proba(X_test)[:, 1]

results_tfidf = {
    'roc_auc': roc_auc_score(y_test, y_pred_proba),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'tuning_time_s': tuning_time_tfidf,
    'train_time_s': train_time_tfidf,
    'total_time_s': tuning_time_tfidf + train_time_tfidf,
    'inference_time_s': inference_time_tfidf,
    'inference_ms_per_tweet': inference_time_tfidf / len(X_test) * 1000
}

for metric, value in results_tfidf.items():
    print(f"{metric}: {value:.3f}")

# Word embeddings como features

## Idea general

En lugar de representar cada tweet como un vector disperso de conteos (TF) o pesos (TF-IDF) sobre el vocabulario, ahora vamos a representarlo como el **promedio de los vectores de embedding preentrenados** de sus palabras. Usamos los embeddings estáticos en español **SBWCE** (`SBW-vectors-300-min5`, ya descargados al inicio del notebook), donde cada palabra se representa con un vector denso de 300 dimensiones.

El preprocesamiento acá es más simple: solo eliminamos URLs y menciones, y reemplazamos números por `DIGITO`. No hace falta sacar puntuación ni acentos porque las palabras que no estén en el vocabulario de los embeddings simplemente se ignoran (*out-of-vocabulary*).

Como en las secciones anteriores, vectorizamos por separado los tweets en español de `train`, `dev` y `test`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import re
import time
from gensim.models import KeyedVectors
import nltk
from nltk.tokenize import word_tokenize
import warnings
warnings.filterwarnings('ignore')

# Descargas necesarias para tokenización
nltk.download('punkt_tab')
nltk.download('punkt')

# Preprocesamiento simple: solo removemos URLs, menciones y reemplazamos dígitos
def preprocess_text_embed(text):
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'\d+', 'DIGITO', text)
    return text

# Cargamos un split y nos quedamos con los tweets en español
def load_hateval_split_embed(path):
    df = pd.read_csv(path)
    df = df[df['language'] == 'es'].reset_index(drop=True)
    df['text_clean'] = df['text'].apply(preprocess_text_embed)
    return df

train_df = load_hateval_split_embed(train_path)
dev_df = load_hateval_split_embed(dev_path)
test_df = load_hateval_split_embed(test_path)

# Cargamos el modelo de word embeddings
def load_embeddings(path):
    print("Cargando embeddings...")
    return KeyedVectors.load_word2vec_format(path, binary=True)

# La carga del modelo (~1GB) es un costo único: se paga una sola vez al
# iniciar el notebook, no por cada modelo que entrenemos con estos vectores
t0 = time.perf_counter()
word_vectors = load_embeddings("./models/SBW-vectors-300-min5.bin")
embed_load_time = time.perf_counter() - t0

# Función para obtener el vector promedio de un tweet
def get_mean_vector(text, word_vectors, vector_size=300):
    words = word_tokenize(text.lower())  # Tokenizamos y convertimos a minúsculas
    word_vectors_list = []

    for word in words:
        try:
            vector = word_vectors[word]
            word_vectors_list.append(vector)
        except KeyError:
            continue  # Ignoramos palabras que no están en el embedding

    if word_vectors_list:
        return np.mean(word_vectors_list, axis=0)
    else:
        return np.zeros(vector_size)  # Vector de ceros si no hay palabras válidas

# Convertimos los tweets de un split a vectores
def vectorize_split(df, word_vectors):
    print("Vectorizando tweets...")
    vectors = [get_mean_vector(text, word_vectors) for text in df['text_clean']]
    X = pd.DataFrame(vectors, columns=[f'V{i+1}' for i in range(300)])
    X['id'] = df['id'].values
    y = df['HS'].values
    return X, y

# Cronometramos la vectorización de train y test por separado: a diferencia
# de TF/TF-IDF, acá la vectorización no vive adentro de un Pipeline, así que
# hay que sumarla a mano al tiempo de entrenamiento/inferencia más abajo
t0 = time.perf_counter()
X_train_embed_full, y_train = vectorize_split(train_df, word_vectors)
vect_time_train = time.perf_counter() - t0

X_dev_embed_full, y_dev = vectorize_split(dev_df, word_vectors)

t0 = time.perf_counter()
X_test_embed_full, y_test = vectorize_split(test_df, word_vectors)
vect_time_test = time.perf_counter() - t0

# Guardamos los ids por separado y nos quedamos solo con las columnas de vectores
train_ids = X_train_embed_full['id']
dev_ids = X_dev_embed_full['id']
test_ids = X_test_embed_full['id']
X_train_embed = X_train_embed_full.drop('id', axis=1)
X_dev_embed = X_dev_embed_full.drop('id', axis=1)
X_test_embed = X_test_embed_full.drop('id', axis=1)

In [ ]:
X_train_embed.head()

## Búsqueda de hiperparámetros usando el split de validación

Repetimos el mismo esquema que en las secciones anteriores: para cada valor de `C` entrenamos sobre `X_train_embed`/`y_train` y evaluamos el ROC AUC sobre `X_dev_embed`/`y_dev`, y nos quedamos con el que mejor performa en `dev`.

A diferencia de TF/TF-IDF, acá la vectorización ya se hizo una sola vez (sección anterior) y se reutiliza en los 30 candidatos, así que el tiempo de esta búsqueda (`tuning_time_s`) va a ser mucho más barato en proporción — lo comparamos en la sección final.

In [ ]:
# Grilla de valores para C
C_values = np.logspace(-10, 1, 30)

val_results = []

In [ ]:
# A diferencia de TF/TF-IDF, acá la vectorización ya se hizo una sola vez
# (celda anterior) y se reutiliza en los 30 candidatos: este tiempo solo
# incluye ajustar la regresión logística, no volver a vectorizar
t0 = time.perf_counter()
print("Buscando el mejor valor de C...")
for C in C_values:
    model = LogisticRegression(
        C=C,
        penalty='l1',
        solver='liblinear',
        random_state=234
    )

    model.fit(X_train_embed, y_train)
    dev_proba = model.predict_proba(X_dev_embed)[:, 1]

    val_results.append({
        'C': C,
        'roc_auc_dev': roc_auc_score(y_dev, dev_proba)
    })
tuning_time_embed = time.perf_counter() - t0

val_results_df = pd.DataFrame(val_results)

best_idx = val_results_df['roc_auc_dev'].idxmax()
best_C_embed = val_results_df.loc[best_idx, 'C']

print(f"\nTiempo de búsqueda (30 valores de C, reutilizando la vectorización): {tuning_time_embed:.1f}s")
print(f"Mejor valor de C: {best_C_embed:.6f}")
print(f"Mejor ROC AUC (dev): {val_results_df.loc[best_idx, 'roc_auc_dev']:.3f}")

In [ ]:
# Ajustamos el modelo final con el mejor C
final_model_embed = LogisticRegression(
    C=best_C_embed,
    penalty='l1',
    solver='liblinear',
    random_state=234
)

t0 = time.perf_counter()
final_model_embed.fit(X_train_embed, y_train)
fit_time_embed = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred = final_model_embed.predict(X_test_embed)
predict_time_embed = time.perf_counter() - t0

y_pred_proba = final_model_embed.predict_proba(X_test_embed)[:, 1]

# Sumamos la vectorización (calculada más arriba) al tiempo del clasificador,
# para que sea comparable con TF y TF-IDF (donde la vectorización ya está
# incluida en el Pipeline)
train_time_embed = vect_time_train + fit_time_embed
inference_time_embed = vect_time_test + predict_time_embed

results_embed = {
    'roc_auc': roc_auc_score(y_test, y_pred_proba),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'tuning_time_s': tuning_time_embed,
    'train_time_s': train_time_embed,
    'total_time_s': tuning_time_embed + train_time_embed,
    'inference_time_s': inference_time_embed,
    'inference_ms_per_tweet': inference_time_embed / len(X_test_embed) * 1000
}

print("\nResultados finales:")
for metric, value in results_embed.items():
    print(f"{metric}: {value:.3f}")

print(f"\n(Costo único, no incluido arriba: carga de SBWCE = {embed_load_time:.1f} s)")

# Comparación de resultados

Además de las métricas de desempeño, comparamos el **costo computacional** de cada representación con estas columnas:

- `tuning_time_s`: tiempo de la búsqueda de hiperparámetros completa (30 valores de `C` probados sobre `dev`).
- `train_time_s`: tiempo de ajustar el **modelo final** (con el mejor `C` ya elegido) sobre `train`.
- `total_time_s`: `tuning_time_s + train_time_s` — el costo real de llegar a un modelo entrenado y elegido, de punta a punta.
- `inference_time_s` / `inference_ms_per_tweet`: tiempo de predecir sobre `test`, normalizado por tweet en el segundo caso.

**La búsqueda de `C` no cuesta lo mismo en las tres representaciones, y la razón es estructural, no una casualidad de esta corrida:**

- En **TF y TF-IDF** el vectorizador vive *adentro* del `Pipeline` de sklearn. Cada uno de los 30 candidatos de `C` llama a `pipeline.fit(...)`, que **re-vectoriza `train` desde cero cada vez**, aunque el vocabulario no cambie entre candidatos. `tuning_time_s` paga ese costo 30 veces.
- En **embeddings**, la vectorización (`vectorize_split`) se hace **una sola vez**, antes del loop de búsqueda; el loop solo reajusta la regresión logística sobre la matriz ya vectorizada. `tuning_time_s` acá es mucho más barato en proporción.

Esto puede invertir la comparación de costo total: aunque un solo `fit` del pipeline de TF/TF-IDF sea rápido, sumar el costo de los 30 candidatos de la búsqueda puede hacer que el costo de punta a punta (`total_time_s`) termine siendo mayor que el de embeddings, donde el trabajo pesado (vectorizar) se paga una única vez y se reutiliza tanto en la búsqueda como en el modelo final.

Para TF y TF-IDF, `train_time_s` e `inference_time_s` salen de cronometrar `fit`/`predict` del `Pipeline`, que ya incluye el vectorizador. Para embeddings no hay un `Pipeline` que junte vectorización y clasificador, así que sumamos a mano el tiempo de vectorizar (`get_mean_vector` sobre cada tweet) al tiempo del `LogisticRegression`, para que las tres filas midan lo mismo.

La carga del modelo SBWCE (~1GB) **no** está incluida en ninguna de estas columnas: es un costo único que se paga una sola vez al iniciar el notebook, no cada vez que entrenamos o tuneamos un modelo con esos vectores.

**Nota:** estos tiempos dependen de la máquina donde se corre el notebook y varían un poco entre corridas — lo que importa acá es el orden de magnitud y la comparación relativa entre representaciones, no el número exacto.

In [ ]:
results_tf_df = pd.DataFrame([results_tf], index=['TF'])
results_tfidf_df = pd.DataFrame([results_tfidf], index=['TF-IDF'])
results_embed_df = pd.DataFrame([results_embed], index=['Word Embeddings'])

comparison_df = pd.concat([results_tf_df, results_tfidf_df, results_embed_df])

print("Comparación de resultados de los modelos (evaluados sobre test):")
display(comparison_df)

print(f"\nCosto único (no incluido en la tabla): carga del modelo SBWCE = {embed_load_time:.1f} s")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter

# Colores fijos por representación (mismo orden/color en los dos paneles)
colors = {
    'TF': '#2a78d6',
    'TF-IDF': '#eb6834',
    'Word Embeddings': '#1baf7a',
}
ink_secondary = '#52514e'
ink_muted = '#898781'
gridline = '#e1e0d9'
surface = '#fcfcfb'

models = list(comparison_df.index)
bar_colors = [colors[m] for m in models]

fig, (ax_train, ax_tradeoff) = plt.subplots(1, 2, figsize=(11, 4.7))
fig.patch.set_facecolor(surface)

# --- Panel izquierdo: costo total, apilado en búsqueda de C + modelo final ---
ax_train.set_facecolor(surface)
ax_train.grid(axis='y', color=gridline, linewidth=1, zorder=0)

tuning_vals = comparison_df['tuning_time_s'].values
train_vals = comparison_df['train_time_s'].values
total_vals = comparison_df['total_time_s'].values

# Segmento inferior: búsqueda de C (30 candidatos, más pesado en TF/TF-IDF
# porque re-vectorizan en cada uno). Segmento superior: modelo final (1 fit).
# El segmento superior usa un tono más claro de la misma familia de color y
# un anillo en el color de superficie para separarse del segmento de abajo.
ax_train.bar(models, tuning_vals, width=0.5, color=bar_colors, zorder=3,
             label='Búsqueda de C (30 candidatos)')
ax_train.bar(models, train_vals, width=0.5, bottom=tuning_vals,
             color=bar_colors, alpha=0.45, zorder=3,
             edgecolor=surface, linewidth=2,
             label='Modelo final (1 fit)')
for x, total in zip(models, total_vals):
    ax_train.text(x, total, f'{total:.1f}s', ha='center', va='bottom',
                   color=ink_secondary, fontsize=10)

# Headroom para que las etiquetas de total no choquen con nada, y leyenda
# debajo del eje x (afuera del área de barras) para que no tape ninguna
ax_train.set_ylim(0, total_vals.max() * 1.18)
ax_train.set_ylabel('segundos', color=ink_secondary)
ax_train.set_title('Costo total: búsqueda de C + modelo final',
                    color='#0b0b0b', fontsize=11)
ax_train.tick_params(colors=ink_muted)
ax_train.legend(loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=2,
                 frameon=False, fontsize=8.5, labelcolor=ink_secondary)
for spine in ('top', 'right', 'left'):
    ax_train.spines[spine].set_visible(False)
ax_train.spines['bottom'].set_color(ink_muted)

# --- Panel derecho: trade-off desempeño vs. costo de inferencia ---
ax_tradeoff.set_facecolor(surface)
ax_tradeoff.grid(color=gridline, linewidth=1, zorder=0)
ax_tradeoff.scatter(comparison_df['inference_ms_per_tweet'], comparison_df['f1'],
                     s=160, c=bar_colors, edgecolor=surface, linewidth=2, zorder=3)
for model in models:
    row = comparison_df.loc[model]
    ax_tradeoff.annotate(model, (row['inference_ms_per_tweet'], row['f1']),
                          textcoords='offset points', xytext=(8, 6),
                          color=ink_secondary, fontsize=10)
ax_tradeoff.set_xscale('log')
ax_tradeoff.xaxis.set_minor_formatter(NullFormatter())  # evita ticks menores amontonados
ax_tradeoff.set_xlabel('ms de inferencia por tweet (escala log)', color=ink_secondary)
ax_tradeoff.set_ylabel('F1 (test)', color=ink_secondary)
ax_tradeoff.set_title('Desempeño vs. costo de inferencia', color='#0b0b0b', fontsize=11)
ax_tradeoff.tick_params(colors=ink_muted)
for spine in ('top', 'right'):
    ax_tradeoff.spines[spine].set_visible(False)
for spine in ('left', 'bottom'):
    ax_tradeoff.spines[spine].set_color(ink_muted)

fig.suptitle('Costo computacional vs. desempeño', color='#0b0b0b', fontsize=13, y=1.03)
fig.text(0.5, -0.06,
         'Los tiempos dependen de la máquina donde corre el notebook: importa el orden de magnitud, no el valor exacto. '
         'La inferencia no incluye el costo de búsqueda de hiperparámetros (solo se paga una vez, al entrenar).',
         ha='center', color=ink_muted, fontsize=9)
fig.tight_layout()
plt.show()

# Conclusiones

Estos son los resultados de una corrida real del notebook (con el SBWCE real, no simulado):

| | ROC AUC | F1 | Precision | Recall | tuning_time_s | train_time_s | total_time_s | inference_ms_per_tweet |
|---|---|---|---|---|---|---|---|---|
| TF | 0.777 | 0.665 | 0.636 | 0.697 | 36.6 | 0.48 | 37.1 | 0.038 |
| TF-IDF | 0.773 | 0.665 | 0.591 | 0.759 | 16.4 | 0.69 | 17.1 | 0.040 |
| Word Embeddings | 0.759 | 0.596 | 0.674 | 0.533 | 22.5 | 10.31 | 32.8 | 0.476 |

(Los valores exactos van a variar entre corridas y máquinas — lo que importa es el patrón, no el número puntual.)

## Desempeño: TF y TF-IDF le ganan a embeddings acá

TF y TF-IDF empatan en F1 (0.665), unos 7 puntos por encima de embeddings (0.596). Tiene sentido para *hate speech*: la señal discriminativa suele estar en palabras o frases puntuales (insultos, *slurs* específicos), y una LASSO sobre 1-2 gramas puede aislar exactamente esos términos con un coeficiente alto. El promedio de embeddings, en cambio, diluye esa señal — mezcla todas las palabras del tweet en un solo vector de 300 dimensiones, perdiendo cuál palabra puntual disparó el odio. Con solo 4500 tweets de entrenamiento tampoco hay tanto margen para que embeddings compense por generalización semántica.

**El trade-off precision/recall es la diferencia más interesante y práctica.** TF-IDF prioriza recall (0.76) a costa de precision (0.59): atrapa más discurso de odio real pero con más falsos positivos. Embeddings hace lo opuesto: alta precision (0.67), recall bajo (0.53) — es conservador, se pierde casi la mitad del odio real. Si el costo de dejar pasar contenido de odio es alto, TF-IDF es preferible; si el costo de un falso positivo es alto (por ejemplo, penalizar cuentas por error), embeddings.

## Costo: la historia es más rica de lo esperado

Tres cosas para destacar:

1. **TF-IDF tunea ~2.2x más rápido que TF** (16.4s vs 36.6s) a pesar de re-vectorizar el mismo vocabulario 30 veces, igual que TF. La diferencia no está en la vectorización sino en el ajuste de LASSO: `TfidfVectorizer` normaliza cada vector (norma L2 por defecto), lo que le da a `liblinear` un problema de optimización mejor condicionado que converge más rápido. `CountVectorizer` deja conteos crudos sin normalizar, con magnitudes mucho más dispares entre features, y eso hace que el solver L1 tarde más en converger.

2. **Embeddings evita re-vectorizar en el tuning (como se esperaba estructuralmente) pero igual termina caro**, porque `train_time_s` (10.3s) está dominado por vectorizar 4500 tweets con el SBWCE real — mucho más costoso que ajustar el clasificador en sí. El resultado neto: el costo total de embeddings (32.8s) queda muy cerca del de TF (37.1s), y **TF-IDF gana claramente como la opción más barata de punta a punta** — ni la representación "simple" (TF) ni la "sofisticada" (embeddings) le ganan.

3. **En inferencia, embeddings sí es el más caro por lejos**: ~0.48ms/tweet vs ~0.04ms/tweet de TF/TF-IDF — unas 12x más lento por tweet. Acá el costo real de embeddings no está en entrenar, está en que cada predicción nueva paga tokenizar + buscar en un vocabulario de ~1M palabras + promediar. (Esto no incluye la carga del modelo SBWCE, ~1GB, que es un costo único aparte de esta tabla.)

## Conclusión práctica

Para esta tarea puntual, con este modelo lineal y esta cantidad de datos, **TF-IDF es la opción dominante**: mismo F1 que TF, mejor recall, y es la más barata tanto para tunear como para servir. Embeddings pierde en las tres dimensiones (desempeño, costo total, costo de inferencia) frente a TF-IDF — un resultado legítimo y pedagógicamente útil: una representación "más sofisticada" no gana automáticamente, sobre todo cuando se la usa de forma simple (promedio de vectores) en una tarea donde palabras puntuales importan más que el significado semántico agregado, y con pocos datos de entrenamiento.

# Ejercicio

Entrenar y tunear un Random Forest usando features construidas con TF-IDF y otro con embeddings, para clasificar los tweets en español del dataset HatEval. Usá el mismo esquema que en este notebook: elegí los hiperparámetros con el split de `dev` y reportá el desempeño final sobre `test`, incluyendo tiempo de búsqueda de hiperparámetros, entrenamiento del modelo final e inferencia, igual que hicimos acá.

¿Cuál resulta más eficiente? ¿Por qué? Tené en cuenta tanto el desempeño (ROC AUC, F1) como el costo computacional de punta a punta (`total_time_s`) que calculamos en la sección anterior — y si el vectorizador que uses vive adentro de un `Pipeline` o no, porque eso cambia cuántas veces se paga el costo de vectorizar durante la búsqueda de hiperparámetros.

Como desafío adicional: ¿qué esperarías que pase si usás estos mismos embeddings en español (SBWCE) para vectorizar los tweets en **inglés** del dataset (`language == 'en'`)?

In [ ]:
###